# jet — halo mass function quick start

`jet.emulator.hmf.HMFEmulator` predicts the cumulative halo abundance
`n(>= M)` and its derivative `dn/dlnM`, in the `RockstarM200m` mass definition,
from eight cosmological parameters.

This notebook walks the whole public surface: the parameter vector, the two
prediction methods, the data vector underneath them, and the plain
`jet.emulator.Emulator` that the physics is wrapped around.

**Before you start.** The bundled weights are not tracked by git, so a fresh
clone has none:

```bash
python tools/build_hmf_bundle.py
```

Everything below runs on NumPy and SciPy alone — no scikit-learn, no PyTorch,
no CAMB.

In [ ]:
import numpy as np

from jet.emulator.hmf import (
    HMFEmulator,
    castro23_cumulative,
    data_slices,
    mass_edges,
    mass_slices,
    theta_spec,
    z_grid,
)

hmf = HMFEmulator.load()
print(type(hmf).__name__)
print("input :", hmf.x_spec.names)
print("output:", hmf.y_spec.name, "->", hmf.y_spec.n_bins, "values")

## The parameter vector

`theta_spec()` is the whole interface: eight named, bounded axes in a fixed
column order. The bounds are the region the reference emulator was trained on —
worth respecting, because outside them the underlying Gaussian process is
extrapolating and will say so with a `RuntimeWarning`.

In [ ]:
for param in theta_spec():
    low, high = param.bounds
    print(f"  {param.name:<8} {low:>10.4g} .. {high:<10.4g}")

A prediction point is a plain `(n_samples, 8)` array in that order — or a
single row. There is no cosmology object to construct and no state to set:
`predict` is a pure function of its input, so the same object can be called
concurrently and any number of times.

In [ ]:
theta = np.array([0.049, 0.31, 67.66, 0.9665, 2.1e-9, -1.0, 0.0, 0.06])

n = hmf.number_density(theta, z=[0.0, 0.5, 1.0], M=[1e12, 1e13, 1e14])
print("shape:", n.shape, "  ->  (n_samples, n_z, n_M)")
print()
print("rows: z = 0.0, 0.5, 1.0    columns: M = 1e12, 1e13, 1e14 Msun/h")
print(n[0])

`n(>= M)` is in `(h/Mpc)^3`. Multiply by a volume in `(Mpc/h)^3` for an
expected halo count. `dndlnM` is the same model differentiated, and takes the
same arguments.

In [ ]:
d = hmf.dndlnM(theta, z=[0.0, 0.5, 1.0], M=[1e12, 1e13, 1e14])
print("dn/dlnM  [(h/Mpc)^3]")
print(d[0])

Both `z` and `M` are interpolated, so any value inside the emulator's range
works. Leaving them out evaluates the native grid the weights are stored on.

In [ ]:
M_grid = np.logspace(11.5, 15.0, 60)
n_at_z0 = hmf.number_density(theta, z=0.0, M=M_grid)[0, 0]

print("cumulative abundance at z = 0")
for mass, density in zip(M_grid[::12], n_at_z0[::12]):
    print(f"  M = {mass:.2e} Msun/h   n(>=M) = {density:.4e} (h/Mpc)^3")

## What is actually being emulated

Not the mass function. The reference emulates the *ratio* of the mass function
to an analytic Castro23 baseline, because that ratio is a smooth order-unity
function of cosmology while the mass function itself spans ten orders of
magnitude. Recovering the physical quantity is a multiplication by a closed
form, which `HMFEmulator` does for you — but both halves are reachable if you
want them.

In [ ]:
ratio = hmf.ratio(theta)
print(f"ratio           : {ratio.shape}, range {ratio.min():.4f} .. {ratio.max():.4f}")

baseline = castro23_cumulative(theta, z=[0.0], M=[1e13])
print(f"Castro23 base   : n(>=1e13) = {baseline[0, 0, 0]:.4e} (Gpc/h)^-3")

The emulated ratio is not a rectangle: the reference trained on a different
mass range at each redshift, and the trained range narrows towards high
redshift. The data vector is those twelve variable-length blocks concatenated,
`z = 0` first — 425 numbers in total. `data_slices()` gives the column ranges,
`mass_slices()` the mass bins they cover.

In [ ]:
blocks, bins, edges = data_slices(), mass_slices(), mass_edges()
for j in (0, 1, len(z_grid()) - 1):
    start, stop = blocks[j]
    low, high = bins[j]
    print(
        f"  z = {z_grid()[j]:<5} data vector [{start:3d}:{stop:3d}]   "
        f"mass bins [{low:2d}:{high:2d}]   "
        f"{edges[low]:.1e} .. {edges[high]:.1e} Msun/h"
    )

## The plain `jet.Emulator` underneath

`HMFEmulator` is a thin layer. The model itself is an ordinary
`jet.emulator.Emulator` — the same class you would get from training on your own
simulations — holding an input transform chain, an output transform chain and a
regression backend. That is what `HMFEmulator` adds the Castro23 baseline and
the interpolation on top of.

In [ ]:
em = hmf.emulator
print(type(em).__name__)
print("  x chain :", [step.name for step in em.x_chain])
print("  y chain :", [step.name for step in em.y_chain])
print("  backend :", em.backend)

mean, std = em.predict(theta, return_std=True)
print(f"\n  predict -> {mean.shape}, std {std.min():.2e} .. {std.max():.2e}")

That shared machinery is the reason a reference emulator and a model you
train yourself behave the same way from the outside:

```python
# A statistic from the bundled reference        # A model trained on your own data
hmf = HMFEmulator.load()                        em = Emulator(x_spec, y_spec, backend="gp", n_pca=20)
hmf.number_density(theta, z=z, M=M)             em.fit(X, y)
hmf.dndlnM(theta, z=z, M=M)                     mean, std = em.predict(X_new)
                                                em.save("wp.gp.npz")
```

### Plotting (optional)

`matplotlib` is not part of jet's dependencies. If you have it, this draws the
same curve the table above printed.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib is not installed; skipping the plot")
else:
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.loglog(M_grid, n_at_z0)
    ax.set(xlabel="M  [Msun/h]", ylabel="n(>=M)  [(h/Mpc)^3]", title="z = 0")
    plt.show()

## Notes

- **Units.** Masses are `Msun/h`; number densities are `(h/Mpc)^3`.
- **Cost.** The Gaussian process is negligible; about a third of a second per
  cosmology goes into the Castro23 baseline, which integrates twelve redshifts
  against four hundred radii. That is why the ratio is what got emulated.
- **Mass definitions.** Only `RockstarM200m` is bundled. The other four the
  reference provides (`FoFM200m`, `FoFM200c`, `RockstarMvir`,
  `RockstarMvir_bound`) differ by eight Castro23 coefficients and their own
  weights.
- **The weights.** `jet/data/hmf_rockstar_m200m.gp.npz`, about 65 kB, resolved
  against `$JET_DATA_DIR` when that is set. See `jet/data/README.md`.
